# Phase 2: Feature Engineering with DistilBERT Text Encoding

In Phase 1 the conversational text was compressed into 20 PCA components, which turned out to be the main bottleneck for the text-based models. Phase 2 drops the PCA step and passes each conversation through a frozen `distilbert-base-uncased` encoder, keeping the full 768-dim `[CLS]` representation.

DistilBERT is about 40% smaller than BERT-base but retains most of its language understanding, so it works well as a fixed feature extractor when we don't want to fine-tune.

Output: `saas_features_pretrained.parquet` — all tabular features plus the 768 DistilBERT columns per conversation. The modeling notebook loads this directly.

In [1]:
!pip install -q datasets transformers torch pandas numpy scikit-learn

In [2]:
# 1. Imports and device setup
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Step 1 — Load and sample the dataset

Same 8,000 conversations Phase 1 used (`random_state=42`), so the train/val/test split will be identical later on and F1 differences should reflect model changes rather than data. The pre-computed embedding columns that ship with the HuggingFace release are dropped since we're producing our own.

In [3]:
# 2. Load and sample dataset (same seed as Phase 1 for direct comparability)
dataset = load_dataset("DeepMostInnovations/saas-sales-conversations", split="train")
df = dataset.to_pandas().sample(n=8000, random_state=42).reset_index(drop=True)

# Drop identifier/JSON columns and pre-existing embeddings (we generate our own)
cols_to_drop = [c for c in df.columns if c.startswith('embedding_')] + \
               ['scenario', 'conversation', 'probability_trajectory', 'conversation_id', 'company_id', 'company_name']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f"Initial shape: {df.shape}")

README.md: 0.00B [00:00, ?B/s]

cleaned_custom_dataset.csv:   0%|          | 0.00/7.17G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Initial shape: (8000, 10)


## Step 2 — Tabular feature engineering

One-hot the categorical columns, compute text-length stats, add a few engagement × length interactions, log-transform the skewed columns, and standardize the continuous ones. This is the same feature pipeline Phase 1 used.

In [4]:
# 3. Tabular feature preparation (same as Phase 1)
from sklearn.preprocessing import StandardScaler

categorical_cols = ['product_name', 'product_type', 'conversation_style', 'conversation_flow', 'communication_channel']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Basic text stats
df['text_length'] = df['full_text'].astype(str).apply(len)
df['word_count'] = df['full_text'].astype(str).apply(lambda x: len(x.split()))
df['average_word_length'] = df['text_length'] / (df['word_count'] + 1)

# Interaction features
df['engagement_x_length'] = df['customer_engagement'] * df['conversation_length']
df['engagement_per_turn'] = df['customer_engagement'] / (df['conversation_length'] + 1)
df['text_per_turn'] = df['text_length'] / (df['conversation_length'] + 1)

# Log transforms on skewed features
skewed_features = ['text_length', 'word_count', 'conversation_length']
for feat in skewed_features:
    df[f'{feat}_log'] = np.log1p(df[feat])

# Standardize continuous features
features_to_scale = ['customer_engagement', 'sales_effectiveness', 'average_word_length',
                     'engagement_x_length', 'engagement_per_turn', 'text_per_turn'] + [f'{f}_log' for f in skewed_features]

scaler = StandardScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print(f"Shape after tabular engineering: {df.shape}")

Shape after tabular engineering: (8000, 77)


## Step 3 — Load the pre-trained text encoder

Load `distilbert-base-uncased` and set `requires_grad=False` on every parameter. We use it as a fixed feature extractor with no fine-tuning, so the encoder stays in `eval()` mode and we only do forward passes through it.

In [5]:
# 4. Load DistilBERT (frozen)
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
encoder.eval()

for param in encoder.parameters():
    param.requires_grad = False

print(f"Loaded {MODEL_NAME}. Hidden size: {encoder.config.hidden_size}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded distilbert-base-uncased. Hidden size: 768


## Step 4 — Encode every conversation

For each conversation we take the `[CLS]` token from the last hidden state of DistilBERT and treat that 768-dim vector as a single semantic representation of the conversation.

- `max_length=512` with truncation — conversations longer than this get cut at the limit.
- `batch_size=32`.
- `torch.no_grad()` + `encoder.eval()` — pure inference, no gradient bookkeeping, no dropout.

In [6]:
# 5. Encode all conversations (frozen forward pass, batched)
texts = df['full_text'].astype(str).tolist()
batch_size = 32
max_length = 512
all_cls = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(device)
        out = encoder(**enc)
        cls_vecs = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_cls.append(cls_vecs)

cls_matrix = np.vstack(all_cls).astype(np.float32)
print(f"DistilBERT [CLS] matrix shape: {cls_matrix.shape}")

Encoding:   0%|          | 0/250 [00:00<?, ?it/s]

DistilBERT [CLS] matrix shape: (8000, 768)


Result: an `(8000, 768)` matrix, one row per conversation. Unlike Phase 1 we keep the full 768 dimensions instead of projecting down to 20 PCA components.

## Step 5 — Merge and persist

The next two cells stack the 768 text columns (`tok_0` … `tok_767`) alongside the tabular features and the `outcome` label, giving an `(8000, 844)` frame that we save to Parquet for the modeling notebook.

In [7]:
# 6. Merge text encoding with tabular features
cls_cols = [f'tok_{i}' for i in range(cls_matrix.shape[1])]
cls_df = pd.DataFrame(cls_matrix, columns=cls_cols)

df = df.drop(columns=['full_text']).reset_index(drop=True)
df = pd.concat([df, cls_df], axis=1)

print(f"Final feature matrix shape: {df.shape}")

Final feature matrix shape: (8000, 844)


In [9]:
# 7. Save for Phase 2 modeling notebook
df.to_parquet("saas_features_pretrained.parquet", index=False)
print("File saved successfully")

tabular_n = df.shape[1] - len(cls_cols) - 1
print(f"Tabular features: {tabular_n} | Text (DistilBERT): {len(cls_cols)} | Target: 1")

df.head()

File saved successfully
Tabular features: 75 | Text (DistilBERT): 768 | Target: 1


,outcome,conversation_length,customer_engagement,sales_effectiveness,product_name_AR Renewable Insights,product_name_AutoLogistics Pro,product_name_BioSync,product_name_DataFlow Pro,product_name_DevOps Automation Suite,product_name_EcoSecure Manager,...,tok_758,tok_759,tok_760,tok_761,tok_762,tok_763,tok_764,tok_765,tok_766,tok_767
0,1,12,0.492069,1.193471,False,False,False,False,False,False,...,0.125517,-0.168442,-0.134466,-0.140256,0.050988,0.233537,-0.252694,-0.158906,0.378616,0.260342
1,1,11,0.919787,1.510406,False,False,False,False,False,False,...,0.076725,-0.245348,-0.171675,-0.064933,0.091748,0.265292,-0.197821,-0.092271,0.351799,0.304714
2,1,10,0.492069,1.193471,False,False,False,False,False,False,...,0.015956,-0.270464,-0.053143,-0.110775,0.045127,0.274825,-0.168448,-0.148016,0.366149,0.256460
3,0,13,-0.363368,-0.708134,False,False,False,False,False,False,...,0.108830,-0.286675,-0.013349,-0.189583,0.078395,0.156878,-0.240779,-0.090930,0.332621,0.391484
4,1,9,0.492069,1.193471,False,True,False,False,False,False,...,0.081164,-0.423531,0.021307,-0.177379,0.081075,0.232082,-0.304646,-0.078090,0.398858,0.390085


### Summary

- Text encoder: frozen `distilbert-base-uncased`, `[CLS]` pooled output (768-dim per conversation).
- Tokenization: max_length=512 with truncation.
- Tabular features: same pipeline as Phase 1.
- No PCA step on the text side — the full 768-dim representation is passed downstream.